In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

In [10]:
df = pd.read_excel(r"C:\Users\Mi\OneDrive\Desktop\Перспективные площадки 08.06.xlsx")

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566 entries, 0 to 565
Data columns (total 44 columns):
 #   Column                                                         Non-Null Count  Dtype         
---  ------                                                         --------------  -----         
 0   ID                                                             566 non-null    int64         
 1   Название ПП                                                    566 non-null    object        
 2   Плановый срок выхода на рынок                                  114 non-null    datetime64[ns]
 3   Площадь участка, га                                            528 non-null    float64       
 4   Общая площадь проекта, кв м                                    304 non-null    object        
 5   Площадь наземной части, кв м                                   32 non-null     float64       
 6   Площадь наземного паркинга, кв м                               1 non-null      float64       
 7  

In [14]:
df['Координаты'].unique()

array(['55.833336, 37.535555', '55.726495, 37.576506',
       '55.881212, 37.624722', '55.734465, 37.69815',
       '55.853142, 37.559444', '55.722915, 37.604588',
       '55.613833, 37.58485', '55.73515, 37.616917',
       '55.752314, 37.515772', '55.852729, 37.568895',
       '55.741921, 37.578559', '55.680722, 37.691827',
       '55.76795, 37.605562', '55.697709, 37.644927',
       '55.76769, 37.590342', '55.748817, 37.651385',
       '55.733117, 37.608937', '55.802971, 37.605665',
       '55.754434, 37.535234', '55.692726, 37.525831',
       '55.7475, 37.811384', '55.681521, 37.630851',
       '55.72778, 37.570048', '55.727734, 37.520249',
       '55.573704, 37.664921', '55.817638, 37.730185',
       '55.688842, 37.476231', '55.776573, 37.573851',
       '55.814403, 37.492573', '55.798455, 37.551583',
       '55.805702, 37.800616', '55.643978, 37.640452',
       '55.685485, 37.558375', '55.662882, 37.48561',
       '55.785508, 37.395931', '55.711161, 37.495215',
       '55.760201, 

In [8]:
df = df['Координаты'].dropna()

In [13]:
# Меняем координаты местами
df['Координаты'] = df['Координаты'].apply(lambda x: ', '.join(x.split(', ')[::-1]))

In [16]:
df[["lat", "lon"]] = (
    df["Координаты"]
    .str.split(",", expand=True)
)

df["lat"] = df["lat"].astype(float)
df["lon"] = df["lon"].astype(float)

In [18]:
print(df[["Координаты", "lat", "lon"]].head())

             Координаты        lat        lon
0  55.833336, 37.535555  55.833336  37.535555
1  55.726495, 37.576506  55.726495  37.576506
2  55.881212, 37.624722  55.881212  37.624722
3   55.734465, 37.69815  55.734465  37.698150
4  55.853142, 37.559444  55.853142  37.559444


In [19]:
projects_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(
        df["lon"],   # X = долгота
        df["lat"]    # Y = широта
    ),
    crs="EPSG:4326"
)

In [33]:
districts = pd.read_csv(r"C:\Users\Mi\Downloads\data-1783586815360.csv")

In [22]:
print(districts["coords"].iloc[0][:200])

[[[55.68403, 37.55226], [55.68408, 37.55232], [55.68428, 37.55256], [55.68432, 37.55261], [55.68435, 37.55265], [55.68504, 37.55353], [55.68549, 37.5541], [55.68555, 37.55417], [55.68561, 37.55425], [


In [36]:
print(districts["district"].iloc[0])

Академический


In [26]:
from shapely.geometry import Point

# Первый полигон
poly = districts_gdf.geometry.iloc[0]

# Координата примерно в Академическом районе
point = Point(37.57, 55.69)

print(poly.contains(point))

NameError: name 'districts_gdf' is not defined

In [29]:
projects = df

In [37]:
import pandas as pd
import geopandas as gpd
import ast
from shapely.geometry import Point, Polygon

# ==========================
# 2. Разбираем координаты проектов
# ==========================

projects[["lat", "lon"]] = (
    projects["Координаты"]
    .str.split(",", expand=True)
)

projects["lat"] = projects["lat"].astype(float)
projects["lon"] = projects["lon"].astype(float)

# ==========================
# 3. Создаем точки
# ==========================

projects_gdf = gpd.GeoDataFrame(
    projects,
    geometry=gpd.points_from_xy(
        projects["lon"],   # X
        projects["lat"]    # Y
    ),
    crs="EPSG:4326"
)

# ==========================
# 4. Создаем полигоны районов
# ==========================

def create_polygon(coord_string):

    coords = ast.literal_eval(coord_string)

    # внешний контур
    ring = coords[0]

    # меняем местами широту и долготу
    ring = [(lon, lat) for lat, lon in ring]

    return Polygon(ring)

districts["geometry"] = districts["coords"].apply(create_polygon)

districts_gdf = gpd.GeoDataFrame(
    districts,
    geometry="geometry",
    crs="EPSG:4326"
)

# ==========================
# 5. Пространственное объединение
# ==========================

result = gpd.sjoin(
    projects_gdf,
    districts_gdf[["district", "ao", "geometry"]],
    how="left",
    predicate="within"
)

# ==========================
# 6. Сохраняем
# ==========================

result = result.drop(columns=["geometry", "index_right"])

result.to_excel("projects_result.xlsx", index=False)

print("Готово!")

Готово!


In [38]:
# Первый полигон
poly = districts_gdf.geometry.iloc[0]

# Координата примерно в Академическом районе
point = Point(37.57, 55.69)

print(poly.contains(point))

True
